# VFL Empire — MuZero GPU Training + Survivorship Bias

**Lord FaithDavid's Prediction Engine**
42K match results · 7K odds · MuZero RL on T4 GPU

## The Wald Survivorship Insight
"During WW2, engineers wanted to reinforce wing armor on returning planes. Statistician Wald said: the planes that came back were hit in non-critical areas. The ones that went down — we never see them."

**Our missed predictions are the downed planes.**
We study them to find where the real weaknesses are.


In [ ]:
import torch, psutil, json, numpy as np, os
from pathlib import Path
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE"}')
if torch.cuda.is_available():
    print(f'GPU Mem: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print(f'CPU: {psutil.cpu_count()} cores | RAM: {psutil.virtual_memory().total/1e9:.1f} GB')


## Setup & Data Load


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
WORK = Path('/content/vfl-empire')
WORK.mkdir(exist_ok=True)
os.chdir(WORK)
print(f'Working in: {WORK}')


In [ ]:
get_ipython().run_cell_magic('time', '', '# Clone MuZero and install\nif not (WORK/"muzero-general").exists():\n    !git clone https://github.com/werner-duvaud/muzero-general.git --depth 1\n%cd muzero-general\n!pip install -q tensorboard numpy psutil 2>&1 | tail -2\nprint("Setup complete")')


In [ ]:
# Load VFL data from GitHub repo
if not (WORK/'vfl-empire-repo').exists():
    !git clone https://github.com/faithdavid/vfl-empire.git vfl-empire-repo --depth 1

DATA = WORK/'vfl-empire-repo'/'data'/'consolidated'
for f in ['all_consolidated_joined.json','all_consolidated_odds.json','all_consolidated_results.json']:
    p = DATA/f
    if p.exists():
        print(f'OK {f}: {p.stat().st_size/1024:.1f} KB')
    else:
        print(f'MISSING: {f}')


## VFL Betting Game (GPU-optimized)


In [ ]:
get_ipython().run_cell_magic('writefile', 'games/vfl_gpu.py',
    '"""VFL Betting Game for MuZero -- GPU-accelerated with full 42K data."""\nimport datetime, pathlib, json, numpy as np, torch, os\nfrom pathlib import Path\nfrom .abstract_game import AbstractGame\n\nclass MuZeroConfig:\n    def __init__(self):\n        self.seed = 0\n        self.max_num_gpus = None\n        self.observation_shape = (6, 1, 1)\n        self.action_space = list(range(4))\n        self.players = list(range(1))\n        self.stacked_observations = 0\n        self.muzero_player = 0\n        self.opponent = None\n        self.num_workers = 4\n        self.selfplay_on_gpu = True\n        self.max_moves = 1\n        self.num_simulations = 50\n        self.discount = 0.997\n        self.temperature_threshold = None\n        self.root_dirichlet_alpha = 0.25\n        self.root_exploration_fraction = 0.25\n        self.pb_c_base = 19652\n        self.pb_c_init = 1.25\n        self.network = \'fullyconnected\'\n        self.support_size = 10\n        self.encoding_size = 128\n        self.fc_representation_layers = [128]\n        self.fc_dynamics_layers = [128]\n        self.fc_reward_layers = [64]\n        self.fc_value_layers = [64]\n        self.fc_policy_layers = [64]\n        rp = pathlib.Path(__file__).resolve().parents[1] / \'results\' / \'vfl_gpu\'\n        self.results_path = rp\n        self.save_model = True\n        self.training_steps = 50000\n        self.batch_size = 256\n        self.checkpoint_interval = 50\n        self.value_loss_weight = 0.25\n        self.train_on_gpu = torch.cuda.is_available()\n        self.optimizer = \'SGD\'\n        self.weight_decay = 1e-4\n        self.momentum = 0.9\n        self.lr_init = 0.01\n        self.lr_decay_rate = 0.75\n        self.lr_decay_steps = 150000\n        self.replay_buffer_size = 50000\n        self.num_unroll_steps = 1\n        self.td_steps = 1\n        self.PER = True\n        self.PER_alpha = 0.5\n        self.use_last_model_value = True\n        self.reanalyse_on_gpu = True\n        self.self_play_delay = 0\n        self.training_delay = 0\n        self.ratio = None\n\n    def visit_softmax_temperature_fn(self, trained_steps):\n        if trained_steps < 500e3:\n            return 1.0\n        elif trained_steps < 750e3:\n            return 0.5\n        else:\n            return 0.25\n\n\nclass Game(AbstractGame):\n    def __init__(self, seed=None):\n        self.env = VFLBetting(seed)\n\n    def step(self, act):\n        return self.env.step(act)\n\n    def to_play(self):\n        return self.env.to_play()\n\n    def legal_actions(self):\n        return self.env.legal_actions()\n\n    def reset(self):\n        return self.env.reset()\n\n    def render(self):\n        self.env.render()\n\n    def human_to_action(self):\n        c = input(\'0=HOME 1=DRAW 2=AWAY 3=SKIP: \')\n        while c not in [str(a) for a in self.legal_actions()]:\n            c = input(\'0-3: \')\n        return int(c)\n\n    def action_to_string(self, a):\n        return {0:\'HOME\',1:\'DRAW\',2:\'AWAY\',3:\'SKIP\'}.get(a,\'?\')\n\n\nclass VFLBetting:\n    def __init__(self, seed):\n        self.random = np.random.RandomState(seed)\n        self.matches = self._load_all_data()\n        print(f\'Training pool: {len(self.matches)} matches\')\n        self.idx = 0\n        self.current_match = None\n\n    def _load_all_data(self):\n        matches = []\n        data_dir = Path(\'/content/vfl-empire/vfl-empire-repo/data/consolidated\')\n        # Source 1: Joined real matches (odds + outcome)\n        jf = data_dir / \'all_consolidated_joined.json\'\n        if jf.exists():\n            for m in json.load(open(jf)):\n                oh, od, oa = m.get(\'odds_h\',0), m.get(\'odds_d\',0), m.get(\'odds_a\',0)\n                out = m.get(\'outcome\')\n                if oh and od and oa and out is not None:\n                    matches.append({\'odds_h\':oh,\'odds_d\':od,\'odds_a\':oa,\'outcome\':out,\n                                    \'home\':m.get(\'home\',\'\'),\'away\':m.get(\'away\',\'\'),\n                                    \'league\':m.get(\'league\',\'\'),\'season\':m.get(\'season\',\'\')})\n        # Source 2: All odds (Poisson simulation)\n        of = data_dir / \'all_consolidated_odds.json\'\n        if of.exists():\n            for m in json.load(open(of)):\n                oh = m.get(\'odds_h\',2.0)\n                matches.append({\'odds_h\':oh,\'odds_d\':m.get(\'odds_d\',3.3),\n                               \'odds_a\':m.get(\'odds_a\',2.8),\'simulate\':True})\n        # Source 3: Results (first 30K with derived odds)\n        rf = data_dir / \'all_consolidated_results.json\'\n        if rf.exists():\n            for r in json.load(open(rf))[:30000]:\n                fh = r.get(\'ft_home\', r.get(\'home_goals\'))\n                fa = r.get(\'ft_away\', r.get(\'away_goals\'))\n                if fh is not None and fa is not None:\n                    out = 0 if fh > fa else (2 if fh < fa else 1)\n                    matches.append({\'simulate\':True,\'odds_h\':2.0,\'odds_d\':3.3,\'odds_a\':2.8,\n                                   \'outcome\':out})\n        if len(matches) < 100:\n            for _ in range(50000):\n                matches.append({\'odds_h\':2.0,\'odds_d\':3.3,\'odds_a\':2.8,\'simulate\':True})\n        return matches\n\n    def _simulate(self, oh, od, oa):\n        ti = 1/oh + 1/od + 1/oa\n        ph, pa = 1/oh/ti, 1/oa/ti\n        hs = ph / (ph + pa) * 2\n        as_ = pa / (ph + pa) * 2\n        hg = self.random.poisson(1.437 * hs)\n        ag = self.random.poisson(1.142 * as_)\n        if hg > ag: return 0\n        elif hg < ag: return 2\n        else: return 1\n\n    def to_play(self):\n        return 0\n\n    def reset(self):\n        self.random.shuffle(self.matches)\n        self.idx = 0\n        self.current_match = self.matches[0]\n        return self.get_observation()\n\n    def step(self, action):\n        m = self.current_match\n        if m.get(\'simulate\'):\n            outcome = self._simulate(m[\'odds_h\'], m[\'odds_d\'], m[\'odds_a\'])\n        else:\n            outcome = m.get(\'outcome\', 0)\n        if action == 3:\n            reward = 0.0\n        elif action == outcome:\n            om = {0: m[\'odds_h\'], 1: m[\'odds_d\'], 2: m[\'odds_a\']}\n            reward = min(om[action] - 1.0, 5.0)\n        else:\n            reward = -1.0\n        self.idx += 1\n        done = self.idx >= len(self.matches)\n        if not done:\n            self.current_match = self.matches[self.idx]\n        return self.get_observation(), reward, done\n\n    def get_observation(self):\n        m = self.current_match\n        oh, od, oa = m[\'odds_h\'], m[\'odds_d\'], m[\'odds_a\']\n        ti = 1/oh + 1/od + 1/oa\n        ph, pd_, pa = 1/oh/ti, 1/od/ti, 1/oa/ti\n        return [\n            np.full((1,1), ph, \'f4\'),\n            np.full((1,1), pd_, \'f4\'),\n            np.full((1,1), pa, \'f4\'),\n            np.full((1,1), oh/10, \'f4\'),\n            np.full((1,1), od/10, \'f4\'),\n            np.full((1,1), oa/10, \'f4\'),\n        ]\n\n    def legal_actions(self):\n        return [0, 1, 2, 3]\n\n    def render(self):\n        m = self.current_match\n        print(f\'{m.get("home","?")} vs {m.get("away","?")}: \'\n              f\'{m["odds_h"]:.2f}/{m["odds_d"]:.2f}/{m["odds_a"]:.2f}\')')


## Train MuZero on GPU (50K steps)

*Runtime > Change runtime type > T4 GPU before running.*


In [ ]:
import sys
sys.path.insert(0, str(WORK/'muzero-general'))
from muzero import MuZero
from games.vfl_gpu import MuZeroConfig

config = MuZeroConfig()
print(f'Training: {config.training_steps} steps')
print(f'Batch: {config.batch_size} | GPU: {config.train_on_gpu}')
print(f'Network: {config.encoding_size}-wide FC | Sims: {config.num_simulations}')

mz = MuZero(config)
mz.train()
print('Training complete!')


## Training Results


In [ ]:
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
import matplotlib.pyplot as plt

result_dir = WORK/'muzero-general'/'results'/'vfl_gpu'
event_files = list(result_dir.rglob('events.out.*'))

if event_files:
    ea = EventAccumulator(str(event_files[0]))
    ea.Reload()
    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    plots = [
        ('3.Loss/1.Total_weighted_loss', axes[0,0], 'Total Loss'),
        ('3.Loss/Value_loss', axes[0,1], 'Value Loss'),
        ('3.Loss/Policy_loss', axes[1,0], 'Policy Loss'),
        ('1.Total_reward/1.Total_reward', axes[1,1], 'Total Reward'),
    ]
    for tag, ax, title in plots:
        evs = ea.Scalars(tag)
        if len(evs) > 1:
            ax.plot([e.step for e in evs[1:]], [e.value for e in evs[1:]])
            ax.set_title(title)
            ax.set_xlabel('Step')
            ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig('/content/training_results.png', dpi=150)
    plt.show()
else:
    print('Training may still be in progress...')


## Survivorship Bias -- Bullet Hole Map

We analyze every missed prediction to find the real weaknesses.


In [ ]:
from collections import Counter

DATA = WORK/'vfl-empire-repo'/'data'/'consolidated'
with open(DATA/'all_consolidated_joined.json') as f:
    joined = json.load(f)
print(f'Loaded {len(joined)} joined matches')

def odds_to_probs(oh, od, oa):
    ti = 1/oh + 1/od + 1/oa
    return 1/oh/ti, 1/od/ti, 1/oa/ti

def pick_fav(oh, od, oa):
    p = odds_to_probs(oh, od, oa)
    return max([(0,p[0]),(1,p[1]),(2,p[2])], key=lambda x: x[1])[0]

misses = []
categories = {'short_fav':0,'mod_fav':0,'long_fav':0}
miss_cat = {'short_fav':0,'mod_fav':0,'long_fav':0}
league_hits = Counter()
league_misses = Counter()
league_total = Counter()
upset_types = Counter()

for m in joined:
    oh = m.get('odds_h', 2.0)
    od = m.get('odds_d', 3.3)
    oa = m.get('odds_a', 2.8)
    fh = m.get('ft_home', m.get('home_goals'))
    fa = m.get('ft_away', m.get('away_goals'))
    out = m.get('outcome')
    if out is None:
        if fh is None:
            continue
        out = 0 if fh > fa else (2 if fh < fa else 1)
    fav = pick_fav(oh, od, oa)
    league = m.get('league', 'unknown')
    fav_odds = [oh, od, oa][fav]
    if fav_odds < 1.5:
        cat = 'short_fav'
    elif fav_odds < 2.5:
        cat = 'mod_fav'
    else:
        cat = 'long_fav'
    categories[cat] += 1
    league_total[league] += 1
    if fav == out:
        league_hits[league] += 1
    else:
        miss_cat[cat] += 1
        league_misses[league] += 1
        upset_types(['home_upset','draw','away_upset'][out]) += 1
        misses.append({'odds':(oh,od,oa),'fav':fav,'actual':out,
                      'cat':cat, 'league':league})

total = len(joined)
n_misses = len(misses)
print()
print('=' * 70)
print('  SURVIVORSHIP BIAS - BULLET HOLE MAP')
print('=' * 70)
print(f'  Total matches:   {total}')
print(f'  Missed (bullets): {n_misses} ({n_misses/total*100:.1f}%)')
print(f'  Favs won:         {total-n_misses} ({(total-n_misses)/total*100:.1f}%)')
print()
print('  BULLET HOLE DENSITY BY FAVORITE ODDS:')
for cat, label in [('short_fav','Short fav <=1.5'),('mod_fav','Mod fav 1.5-2.5'),('long_fav','Long fav >2.5')]:
    c = miss_cat[cat]
    t = categories[cat]
    pct = c/t*100 if t else 0
    bar = chr(9608)*int(pct/2) + chr(9617)*(20-int(pct/2))
    print(f'  {label:>22s}: {c:4d}/{t:4d} {bar} {pct:.1f}% miss rate')
print()
print('  UPSET TYPE DISTRIBUTION:')
for ut, c in upset_types.most_common():
    print(f'  {ut:>20s}: {c:4d} ({c/n_misses*100:.1f}%)')
print()
print('  LEAGUE BULLET HOLE DENSITY (top 15):')
for league, t in league_total.most_common(15):
    h = league_hits.get(league, 0)
    m = league_misses.get(league, 0)
    tot = h + m
    if tot >= 10:
        pct = m/tot*100
        bar = chr(9608)*int(pct/2) + chr(9617)*(20-int(pct/2))
        print(f'  {league[:22]:>22s}: {m:4d}/{tot:4d} {bar} {pct:.1f}%')

analysis = {
    'total_matches': total,
    'misses': n_misses,
    'miss_rate': round(n_misses/total, 4) if total else 0,
    'insight': 'Survivorship bias: bullet holes are our missed predictions'
}
with open('/content/survivorship_analysis.json', 'w') as f:
    json.dump(analysis, f, indent=2)
print()
print('Survivorship analysis saved to Drive')


## Evaluate & Save Model


In [ ]:
import sys, shutil
sys.path.insert(0, str(WORK/'muzero-general'))
from muzero import MuZero
from games.vfl_gpu import MuZeroConfig

config = MuZeroConfig()
mz = MuZero(config)

result_dir = WORK/'muzero-general'/'results'/'vfl_gpu'
checkpoints = list(result_dir.glob('*.checkpoint'))
if checkpoints:
    latest = max(checkpoints, key=lambda p: p.stat().st_mtime)
    print(f'Loading: {latest.name}')
    mz.load_model(str(latest))

    n_tests = 2000
    correct = 0
    skips = 0
    profit = 0.0
    for i in range(n_tests):
        game = config.new_game()
        obs = game.reset()
        action = mz.get_action(obs)
        if action == 3:
            skips += 1
        obs, reward, done = game.step(action)
        if reward > 0:
            correct += 1
            profit += reward
        elif reward < 0:
            profit -= 1
        if done:
            break

    print()
    print('=' * 50)
    print(f'  EVALUATION ({n_tests} matches)')
    print('=' * 50)
    bets = n_tests - skips
    if bets > 0:
        print(f'  Correct: {correct}/{bets} ({correct/bets*100:.1f}% when betting)')
    print(f'  Skips:   {skips}/{n_tests} ({skips/n_tests*100:.1f}%)')
    print(f'  Profit:  {profit:.1f} units')
    print(f'  ROI:     {profit/n_tests*100:.2f}%')

    drive_dir = Path('/content/drive/MyDrive/vfl_empire_models')
    drive_dir.mkdir(exist_ok=True, parents=True)
    for cp in checkpoints:
        shutil.copy2(str(cp), str(drive_dir/cp.name))
    src = '/content/survivorship_analysis.json'
    if os.path.exists(src):
        shutil.copy2(src, str(drive_dir/'survivorship_analysis.json'))
    print(f'Saved to Drive: {drive_dir}')
else:
    print('No checkpoint found. Training may still be running.')


## Complete

Model + survivorship analysis saved to **Google Drive** (`vfl_empire_models/`).

**Next steps:**
1. Review the bullet hole analysis -- which leagues/odds ranges kill us?
2. Deploy trained model back to prediction pipeline
3. Retrain focused on high-miss-rate segments
